In [1]:
import pandas as pd
import io
from datetime import datetime, timedelta


In [2]:
# Load check-in records
df = pd.read_csv('FSQ_SG_2013_Checkins.csv', header=None,
                 names=["user_id", "poi_id", "utc_time", "tz_offset"])

# Load POI ID to (lat, lon) mapping
poi_coords = pd.read_csv('FSQ_SG_2013_POI.csv', header=None, 
                         names=["poi_id", "poi_name", "lat", "lon", "cat_name", "country_code"])

In [3]:
# Apply 5-core filtering
def five_core_filter(df, user_col='user_id', item_col='poi_id', min_interactions=5):
    while True:
        user_counts = df[user_col].value_counts()
        item_counts = df[item_col].value_counts()

        valid_users = user_counts[user_counts >= min_interactions].index
        valid_items = item_counts[item_counts >= min_interactions].index

        df_filtered = df[df[user_col].isin(valid_users) & df[item_col].isin(valid_items)]

        if len(df_filtered) == len(df):
            break
        df = df_filtered

    return df_filtered

df_5core = five_core_filter(df)

#  Filter POI metadata to match filtered POIs
filtered_pois = df_5core['poi_id'].unique()
poi_coords_filtered = poi_coords[poi_coords['poi_id'].isin(filtered_pois)]

len(df_5core), len(poi_coords_filtered)


(308825, 9756)

In [4]:
poi_dict = {}
for idx, row in poi_coords_filtered.iterrows():
    poi_dict[row['poi_id']] = {'name': row['poi_name'], 'lat': row['lat'], 'lon':row['lon'], 'cat': row['cat_name']}

len(poi_dict), poi_dict
    

(9756,
 {'4a5eb95bf964a52019bf1fe3': {'name': 'Lucasfilm Animation Singapore',
   'lat': 1.335005,
   'lon': 103.964682,
   'cat': 'Office'},
  '4a73e804f964a52099dd1fe3': {'name': 'Buddha Tooth Relic Temple & Museum',
   'lat': 1.281391,
   'lon': 103.844348,
   'cat': 'Temple'},
  '4ac7f7f1f964a520e9ba20e3': {'name': 'New Majestic Hotel',
   'lat': 1.279386,
   'lon': 103.840514,
   'cat': 'Hotel'},
  '4ac8148ff964a5208bbb20e3': {'name': 'Sri Mariamman Temple',
   'lat': 1.282644,
   'lon': 103.845286,
   'cat': 'Temple'},
  '4afa194af964a520b71622e3': {'name': 'Clarke Quay Central',
   'lat': 1.28897,
   'lon': 103.846958,
   'cat': 'Mall'},
  '4b05880af964a5208aad22e3': {'name': 'Carlton Hotel',
   'lat': 1.295568,
   'lon': 103.852641,
   'cat': 'Hotel'},
  '4b05880af964a52092ad22e3': {'name': 'Hotel Grand Central',
   'lat': 1.301201,
   'lon': 103.841561,
   'cat': 'Hotel'},
  '4b05880af964a52093ad22e3': {'name': 'Hotel Miramar',
   'lat': 1.288553,
   'lon': 103.837152,
   'cat

In [5]:
# Convert to datetime and apply offset
def parse_local_time(utc_str, offset_min):
    dt = datetime.strptime(utc_str, "%a %b %d %H:%M:%S %z %Y")
    offset = timedelta(minutes=offset_min)
    local_dt = dt + offset
    return local_dt.strftime("%Y-%m-%d %H:%M:%S")

df_5core['local_time'] = df_5core.apply(lambda row: parse_local_time(row['utc_time'], row['tz_offset']), axis=1)


In [6]:
df = df_5core.merge(poi_coords_filtered, on='poi_id', how='left')  # assumes 'lat', 'lon' columns exist
df

,user_id,poi_id,utc_time,tz_offset,local_time,poi_name,lat,lon,cat_name,country_code
0,48739,4cda6665d54954811cdc35b2,Tue Apr 03 18:10:51 +0000 2012,480,2012-04-04 02:10:51,NaN,1.340157,103.920199,Home (private),SG
1,21418,4b138667f964a5208b9723e3,Tue Apr 03 18:19:55 +0000 2012,480,2012-04-04 02:19:55,Xin Wang Hong Kong Café,1.359951,103.884701,Chinese Restaurant,SG
2,49530,4f657a99e4b0c65cc5414938,Tue Apr 03 18:22:10 +0000 2012,480,2012-04-04 02:22:10,Ellyland,1.363290,103.965668,Nightclub,SG
3,224175,4d3b30a336718eec3454748e,Tue Apr 03 18:27:03 +0000 2012,480,2012-04-04 02:27:03,NaN,1.358692,103.955007,Home (private),SG
4,82310,4b1cecc1f964a520800a24e3,Tue Apr 03 18:31:03 +0000 2012,480,2012-04-04 02:31:03,McDonald's,1.312430,103.923304,Fast Food Restaurant,SG
...,...,...,...,...,...,...,...,...,...,...
308820,18277,4b1459aef964a520a7a123e3,Mon Sep 16 23:12:56 +0000 2013,480,2013-09-17 07:12:56,Kovan MRT Station (NE13),1.360305,103.885415,Subway,SG
308821,18277,4c7b131e76ce9c74aa63b50c,Mon Sep 16 23:13:18 +0000 2013,480,2013-09-17 07:13:18,Bus Stop 63039 (Kovan Stn Exit C),1.360326,103.885316,Bus Station,SG
308822,9860,4e7f0cf5f790845961c17293,Mon Sep 16 23:13:57 +0000 2013,480,2013-09-17 07:13:57,Mount Vernon Road,1.342152,103.881066,Field,SG
308823,13358,4d9f5b7d3008dcb3dd23b431,Mon Sep 16 23:20:19 +0000 2013,480,2013-09-17 07:20:19,School of Science and Technology (SST),1.313052,103.773508,High School,SG


In [7]:
# Group and format
def format_checkins(group):
    return '|'.join(f"{row.poi_id},{row.lat},{row.lon},{row.local_time}" for row in group.itertuples())

user_checkins = df.groupby('user_id').apply(format_checkins).reset_index()
user_checkins.columns = ['user_id', 'checkins']

C:\Users\admin\AppData\Local\Temp\ipykernel_46100\468531634.py:5: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  user_checkins = df.groupby('user_id').apply(format_checkins).reset_index()


In [8]:
user_checkins.to_csv('./user_checkins.txt', sep='\t', index=False, header=False)


## process the checkin

In [9]:
user_checkins

,user_id,checkins
0,16,"4e94f6845c5c3201e8ea90f0,1.31079,103.72052,201..."
1,51,"4ba4c111f964a520f5b138e3,1.30667,103.832735,20..."
2,73,"4b9096c6f964a5203b9133e3,1.315508,103.8726,201..."
3,126,"4ca221cf8afca093c12e2116,1.339618,103.983224,2..."
4,142,"4c6ddd944d24b60cea51d7d8,1.265771,103.860669,2..."
...,...,...
4649,266489,"4c0272fcf423a593fac0ce16,1.350936,103.988256,2..."
4650,266649,"4bb9b8627421a5939df3c240,1.304113,103.833655,2..."
4651,266683,"4c8e3a94d68c6dcb4059faa1,1.396987,103.910456,2..."
4652,266723,"4b9f97eff964a520a12d37e3,1.324185,103.930063,2..."


In [10]:
user_trajs = {}
min_len, max_len = 5, 200

for _, row in user_checkins.iterrows():
    traj_set = []
    user_id = row['user_id']
    checkin_list = row['checkins'].split('|')
    
    if len(checkin_list) < min_len: 
        continue
    elif len(checkin_list) > max_len:
        trajs = [checkin_list[max(i - max_len, 0):i] for i in range(len(checkin_list), 0, -max_len)]
        trajs = trajs[::-1]
        trajs = [t for t in trajs if len(t)>=min_len]
        traj_set.extend(trajs)
    else:
        traj_set.append(checkin_list)

    
    if len(traj_set) > 0:
        user_trajs[user_id] = traj_set

In [11]:
user_checkins['num_checkins'] = user_checkins['checkins'].apply(lambda x: len(str(x).split('|')))
user_checkins = user_checkins.sort_values(by='num_checkins', ascending=False).reset_index(drop=True)

In [12]:
user_checkins.head(20)

,user_id,checkins,num_checkins
0,73,"4b9096c6f964a5203b9133e3,1.315508,103.8726,201...",2200
1,143,"4b1537bbf964a5206ea923e3,1.294471,103.805967,2...",2051
2,248,"4ba587fef964a520d40f39e3,1.321624,103.885401,2...",1584
3,412,"4b8b40e6f964a5208d9932e3,1.325901,103.950987,2...",1482
4,221,"4c8d9fc2509e3704b8d03d55,1.358358,103.959568,2...",1475
5,377,"4cc2ad3f914137046410b655,1.345119,103.955474,2...",1405
6,16,"4e94f6845c5c3201e8ea90f0,1.31079,103.72052,201...",1295
7,322,"4b0bd124f964a520e03323e3,1.356685,103.988471,2...",1280
8,848,"4b058816f964a520f0b022e3,1.352775,103.944939,2...",1193
9,912,"4c4cc737a34b2d7f76883f98,1.355575,103.869633,2...",1145


In [13]:
len(user_trajs[73]), len(user_trajs[377]), len(user_trajs[912])

(11, 8, 6)

In [14]:
user_trajs[73]

[['4b9096c6f964a5203b9133e3,1.315508,103.8726,2012-04-04 10:59:39',
  '4b2c0d8ff964a52086c024e3,1.31923,103.86157,2012-04-04 21:16:24',
  '4c061b6491d776b0655cf8f9,1.313242,103.871427,2012-04-04 21:39:48',
  '4c2c495db34ad13a901febce,1.314427,103.872597,2012-04-04 21:43:46',
  '4bd4d1066798ef3b357b628d,1.316323,103.873458,2012-04-04 21:48:09',
  '4b9f189df964a5207a1337e3,1.32169,103.873803,2012-04-06 23:29:39',
  '4b9f189df964a5207a1337e3,1.32169,103.873803,2012-04-07 23:35:55',
  '4b78c916f964a5204ce22ee3,1.305082,103.881258,2012-04-08 11:12:59',
  '4b9f189df964a5207a1337e3,1.32169,103.873803,2012-04-08 11:24:41',
  '4be21632b02ec9b6cf364cc0,1.314701,103.872087,2012-04-08 14:33:44',
  '4f13bb64e4b000502e5db842,1.31483,103.871881,2012-04-08 14:34:16',
  '4b058819f964a520d4b122e3,1.321418,103.845949,2012-04-08 14:56:02',
  '4b9f189df964a5207a1337e3,1.32169,103.873803,2012-04-08 22:27:33',
  '4b9f189df964a5207a1337e3,1.32169,103.873803,2012-04-09 23:40:19',
  '4bcbe0333740b7139a516365,1.

In [15]:
num_traj = 0
num_user = 0
for user, trajs in user_trajs.items():
    if len(trajs) > 0:
        num_user += 1
        num_traj += len(trajs)
num_user, num_traj

(4654, 5073)

In [16]:
num_records = user_checkins['num_checkins'].sum()

num_records

np.int64(308825)

In [17]:
user_traj_num = {}
for user, trajs in user_trajs.items():
    num = len(trajs)
    print(user, num)
    if num not in user_traj_num:
        user_traj_num[num] = 1
    else:
        user_traj_num[num] += 1

user_traj_num = dict(sorted(user_traj_num.items(), key=lambda item: item[0]))
user_traj_num

16 7
51 1
73 11
126 4
142 1
143 11
151 1
208 1
210 2
221 8
222 1
237 1
248 8
289 2
296 2
310 3
322 7
348 1
354 4
377 8
412 8
437 4
467 5
559 1
574 1
623 1
653 1
667 1
683 1
800 3
814 1
848 6
888 2
907 1
912 6
957 4
984 4
994 1
1011 2
1014 1
1035 2
1041 1
1054 6
1055 1
1072 1
1129 1
1164 6
1212 1
1256 6
1304 4
1317 1
1326 1
1402 3
1410 4
1470 5
1495 1
1515 1
1543 1
1565 1
1606 1
1610 1
1623 2
1724 1
1760 1
1767 1
1772 2
1776 1
1785 1
1865 3
1889 1
1925 1
1929 1
1986 1
1988 1
1997 4
2008 1
2062 1
2108 1
2113 1
2126 3
2135 2
2159 1
2189 1
2219 1
2228 3
2234 1
2245 1
2252 5
2288 1
2305 1
2306 1
2317 2
2364 1
2371 4
2390 1
2391 4
2400 4
2445 1
2504 1
2537 1
2594 1
2619 1
2662 1
2736 3
2739 1
2746 1
2768 4
2770 3
2789 1
2820 1
2862 3
2877 2
2924 1
2941 4
2967 3
2978 3
3015 1
3033 1
3034 1
3040 1
3075 1
3114 1
3154 2
3172 1
3199 2
3209 1
3260 1
3307 1
3349 4
3361 1
3364 1
3378 1
3460 1
3461 1
3476 1
3516 1
3519 1
3520 1
3524 1
3531 1
3544 1
3559 1
3579 1
3591 1
3594 1
3606 1
3671 2
3749 1
376

{1: 4388, 2: 196, 3: 36, 4: 18, 5: 3, 6: 5, 7: 2, 8: 4, 11: 2}

In [18]:
out_file = "data_200.txt"
out_stat_file = './statics_out.txt'


with io.open(out_file, 'w', encoding='utf8') as o:
    for user in user_trajs:
        traj_seq = ['|'.join([c for c in t]) for t
                    in user_trajs[user]]
        if len(traj_seq) > 0:
            for traj in traj_seq:
                o.write(str(user) + '\t' + traj + '\n')
    o.flush()

with io.open(out_stat_file, 'w', encoding='utf8') as o:
    length_dist = dict()
    for user in user_trajs:
        for t in user_trajs[user]:
            length = len(t)
            if length not in length_dist:
                length_dist[length] = 1
            else:
                length_dist[length] += 1
    length_array = sorted(length_dist.items(), key=lambda x:x[0])
    for length,freq in length_array:
        o.write(str(length) + '\t' + str(freq) + '\n')
    o.flush()

In [19]:
len(poi_dict), poi_dict

(9756,
 {'4a5eb95bf964a52019bf1fe3': {'name': 'Lucasfilm Animation Singapore',
   'lat': 1.335005,
   'lon': 103.964682,
   'cat': 'Office'},
  '4a73e804f964a52099dd1fe3': {'name': 'Buddha Tooth Relic Temple & Museum',
   'lat': 1.281391,
   'lon': 103.844348,
   'cat': 'Temple'},
  '4ac7f7f1f964a520e9ba20e3': {'name': 'New Majestic Hotel',
   'lat': 1.279386,
   'lon': 103.840514,
   'cat': 'Hotel'},
  '4ac8148ff964a5208bbb20e3': {'name': 'Sri Mariamman Temple',
   'lat': 1.282644,
   'lon': 103.845286,
   'cat': 'Temple'},
  '4afa194af964a520b71622e3': {'name': 'Clarke Quay Central',
   'lat': 1.28897,
   'lon': 103.846958,
   'cat': 'Mall'},
  '4b05880af964a5208aad22e3': {'name': 'Carlton Hotel',
   'lat': 1.295568,
   'lon': 103.852641,
   'cat': 'Hotel'},
  '4b05880af964a52092ad22e3': {'name': 'Hotel Grand Central',
   'lat': 1.301201,
   'lon': 103.841561,
   'cat': 'Hotel'},
  '4b05880af964a52093ad22e3': {'name': 'Hotel Miramar',
   'lat': 1.288553,
   'lon': 103.837152,
   'cat

In [20]:
poi_dict_filter = {}
for user, trajs in user_trajs.items():
    for traj in trajs:
        for r in traj:
            r = r.split(',')
            idx = r[0]
            if idx not in poi_dict_filter:
                poi = poi_dict[idx]
                poi_dict_filter[idx] = {'name': poi['name'], 'lat': poi['lat'], 'lon': poi['lon'], 'cat': poi['cat']}

len(poi_dict_filter) 
            

9756

In [21]:
with open('./poi_traj.txt', 'w', encoding='utf-8') as f:
    for poi_id, data in poi_dict.items():
        line = f"{poi_id}\t{data['name']}\t{data['lat']}\t{data['lon']}\t{data['cat']}\n"
        f.write(line)